### 🧪 Exploratory Testing Notebook - RAG Chatbot

### This notebook tests OCR loading, embedding generation and RAG using LangChain + Flan-T5


## ⚙️ Etapas del Proyecto

#### 📁 1. Configurar entorno y paths

In [1]:
import sys
import os
sys.path.append(os.path.abspath(".."))

#### 📄 2. OCR: extraer texto del PDF a chunks

In [2]:
from src.loader import ocr_pdf_to_text_chunks


chunks = ocr_pdf_to_text_chunks("../data/Spain_unicaja_Fixed_Mortage.pdf")
assert len(chunks) > 0, "❌ No se extrajo texto del PDF."
print(f"✅ {len(chunks)} chunks generados.")
print(chunks[0])


✅ OCR extraído y dividido en 10 chunks
✅ 10 chunks generados.
Unicaja Banco, S.A., Avda. Andalucia 10 - 12, Malaga. Inscrito en el Registro Mercantil de Malaga, Tomo 4.952, Libro 3.859, Seccidn 8, Hoja MA-111.580, Folio 1, Inscripcidn 1°. N.I.F. 493139053.

© Unicaja Banco, S.A. Todos los derechos reservados. Prohibida la reproduccion total o parcial por cualquier medio fisico o digital sin autorizacidn expresa de Unicaja Banco, S.A.

| Unicaja

Tu hipoteca 100% online
en 4 sencillos pasos

Simulacion

" Te informamos que tienes disponible en la Web la inf


#### 🧠 4. Crear base de embeddings

In [3]:
from langchain.schema import Document
from src.embeddings import embed_and_store_documents


docs = [Document(page_content=chunk) for chunk in chunks]
db = embed_and_store_documents(docs)

/Users/luisdotto/Documents/CODE/Projects/Information_Technology/GenerativeAI-Banking-RAG-Chatbot/Lang/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/luisdotto/Documents/CODE/Projects/Information_Technology/GenerativeAI-Banking-RAG-Chatbot/src/embeddings.py:23: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name=model_name)
/Users/luisdotto/Documents/CODE/Projects/Information_Technology/GenerativeAI-Banking-RAG

✅ Chroma DB created and saved in 'db/chroma/'


/Users/luisdotto/Documents/CODE/Projects/Information_Technology/GenerativeAI-Banking-RAG-Chatbot/src/embeddings.py:29: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectorstore.persist()


#### 🔗 5. Crear la cadena RAG con Flan-T5

In [4]:
from src.rag_chain import build_rag_chain

qa = build_rag_chain(model_id="google/flan-t5-base") 

/Users/luisdotto/Documents/CODE/Projects/Information_Technology/GenerativeAI-Banking-RAG-Chatbot/src/rag_chain.py:27: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectordb = Chroma(
Device set to use cpu


✅ RAG chain with custom prompt ready


#### ❓ 6. Ask a question and inspect answer + context + latency

In [5]:
import time

query = "What are the terms of the fixed mortgage?"

start = time.time()
response = qa.invoke({"query": query})
end = time.time()

print("🧠 Respuesta:")
print(response["result"])
print(f"⏱️ Tiempo de respuesta: {round(end - start, 2)} segundos")

# Show retrieved chunks
docs = qa.retriever.get_relevant_documents(query)

print("\n📄 Documentos recuperados (top_k):")
for i, doc in enumerate(docs):
    print(f"\n--- Chunk {i+1} ---\n{doc.page_content[:300]}...")

Token indices sequence length is longer than the specified maximum sequence length for this model (851 > 512). Running this sequence through the model will result in indexing errors


🧠 Respuesta:
El importe del trabajo no podra super
⏱️ Tiempo de respuesta: 1.09 segundos

📄 Documentos recuperados (top_k):

--- Chunk 1 ---
tu hipoteca (forma de pago, plazo, cuota...) y te pedimos que
aportes la documentacion necesaria para el estudio de la solicitud.

Necesitaremos, entre otros, estos documentos en formato digital por cada titular:

« Documento de identidad.

- Informe de Vida Laboral (Seguridad Social).

- Si tienes ...

--- Chunk 2 ---
tu hipoteca (forma de pago, plazo, cuota...) y te pedimos que
aportes la documentacion necesaria para el estudio de la solicitud.

Necesitaremos, entre otros, estos documentos en formato digital por cada titular:

« Documento de identidad.

- Informe de Vida Laboral (Seguridad Social).

- Si tienes ...

--- Chunk 3 ---
tu hipoteca (forma de pago, plazo, cuota...) y te pedimos que
aportes la documentacion necesaria para el estudio de la solicitud.

Necesitaremos, entre otros, estos documentos en formato digital por cada titular:

« D

/var/folders/wb/bnxpdw0d06b47nlv4jh8z_1w0000gn/T/ipykernel_31644/3591110074.py:14: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  docs = qa.retriever.get_relevant_documents(query)
